## 1. Configuration et Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
from datetime import datetime

In [ ]:
# Création de la session Spark avec support Delta Lake
spark = SparkSession.builder \
    .appName("IoT_JSON_to_Delta_Bronze") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoints") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 2. Définition du Schéma des Données IoT

Les données proviennent de capteurs intelligents dans des bâtiments :
- `timestamp` : Horodatage de la mesure (ISO 8601)
- `device_id` : Identifiant unique du capteur (ex: sensor-temp-001)
- `building` : Identifiant du bâtiment (A, B, C)
- `floor` : Étage du capteur
- `type` : Type de mesure (temperature, humidity, co2)
- `value` : Valeur mesurée
- `unit` : Unité de mesure (°C, %, ppm)

In [ ]:
# Schéma des données de capteurs IoT
sensor_schema = StructType([
    StructField("timestamp", StringType(), False),
    StructField("device_id", StringType(), False),
    StructField("building", StringType(), False),
    StructField("floor", IntegerType(), False),
    StructField("type", StringType(), False),
    StructField("value", DoubleType(), False),
    StructField("unit", StringType(), False)
])

## 3. Configuration des Chemins

Organisation des répertoires selon l'architecture Médaillon :
- **Source** : Répertoire contenant les fichiers JSON en entrée
- **Bronze** : Données brutes stockées dans Delta Lake
- **Checkpoint** : Point de sauvegarde pour la tolérance aux pannes

In [ ]:
# Configuration des chemins
BASE_PATH = "/tmp/iot_pipeline"
SOURCE_PATH = "data/sensor_data"  # Using existing data folder
BRONZE_PATH = f"{BASE_PATH}/delta/bronze/sensor_data"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_sensor_data"

# Création des répertoires de sortie si nécessaire
os.makedirs(os.path.dirname(BRONZE_PATH), exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print(f"Source Path: {SOURCE_PATH}")
print(f"Bronze Path: {BRONZE_PATH}")
print(f"Checkpoint Path: {CHECKPOINT_PATH}")

## 4. Données Source

Utilisation des données réelles du dossier `data/sensor_data/` contenant ~100 fichiers JSON avec des mesures de capteurs IoT (température, humidité, CO2).

Structure des données :
```json
{"timestamp": "2025-01-12T08:18:00Z", "device_id": "sensor-co2-020", "building": "B", "floor": 1, "type": "co2", "value": 862, "unit": "ppm"}
```

## 5. Lecture du Flux JSON

Utilisation de `readStream` pour lire les fichiers JSON de manière continue

In [ ]:
# Lecture du flux de données JSON
raw_stream = spark.readStream \
    .format("json") \
    .schema(sensor_schema) \
    .option("maxFilesPerTrigger", 1) \
    .load(SOURCE_PATH)

print("Flux de lecture configuré")
print(f"Schema: {raw_stream.schema}")

## 6. Transformations Bronze

Transformations basiques appliquées aux données brutes :
- Ajout d'une colonne `event_timestamp` (conversion du timestamp ISO)
- Ajout d'une colonne `ingestion_time` pour tracer l'horodatage d'ingestion
- Filtrage des valeurs nulles critiques
- Détection d'anomalies basée sur des seuils:
  - CO2 > 1000 ppm
  - Température < 15°C ou > 30°C
  - Humidité < 20% ou > 80%
- Projection des colonnes pertinentes

In [ ]:
# Transformations Bronze : nettoyage et enrichissement
bronze_stream = raw_stream \
    .filter(col("device_id").isNotNull()) \
    .filter(col("building").isNotNull()) \
    .filter(col("timestamp").isNotNull()) \
    .withColumn("event_timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("anomaly_detected", 
                when((col("type") == "co2") & (col("value") > 1000), True)
                .when((col("type") == "temperature") & ((col("value") < 15) | (col("value") > 30)), True)
                .when((col("type") == "humidity") & ((col("value") < 20) | (col("value") > 80)), True)
                .otherwise(False)) \
    .select(
        "device_id",
        "building",
        "floor",
        "type",
        "value",
        "unit",
        "event_timestamp",
        "anomaly_detected",
        "ingestion_time"
    )

print("Transformations Bronze configurées")

## 7. Écriture dans Delta Lake (Bronze)

Configuration de l'écriture en streaming vers Delta Lake :
- **Format** : Delta Lake pour ACID et versioning
- **Mode de sortie** : `append` (ajout des nouvelles données)
- **Checkpoint** : Garantie de tolérance aux pannes
- **Trigger** : `processingTime` pour traiter les données toutes les 10 secondes

In [ ]:
# Écriture en streaming vers Delta Lake Bronze
query = bronze_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime="10 seconds") \
    .start(BRONZE_PATH)

print(f"Streaming query démarré: {query.name}")
print(f"Query ID: {query.id}")
print(f"Status: {query.status}")

## 8. Surveillance du Flux

Vérification de l'état du streaming et des métriques

In [ ]:
# Afficher le statut du streaming
import time

for i in range(3):
    time.sleep(5)
    print(f"\n--- Itération {i+1} ---")
    print(f"Status: {query.status}")
    print(f"Recent Progress:")
    if query.recentProgress:
        latest = query.recentProgress[-1]
        print(f"  - Batch: {latest.get('batchId', 'N/A')}")
        print(f"  - Num Input Rows: {latest.get('numInputRows', 0)}")
        print(f"  - Input Rows Per Second: {latest.get('inputRowsPerSecond', 0)}")
        print(f"  - Process Rows Per Second: {latest.get('processedRowsPerSecond', 0)}")

## 9. Monitoring du Streaming

Les données sont lues automatiquement depuis `data/sensor_data/`. Le streaming traite les fichiers un par un.

In [ ]:
# Attendre le traitement et vérifier le statut
time.sleep(20)
print(f"\n--- Statut du Streaming ---")
print(f"Is Active: {query.isActive}")
print(f"Recent Progress Count: {len(query.recentProgress)}")
if query.recentProgress:
    latest = query.recentProgress[-1]
    print(f"Input Rows: {latest.get('numInputRows', 0)}")

## 10. Vérification des Données Bronze

Lecture batch des données Delta Lake pour validation

In [ ]:
# Lire les données Bronze en mode batch
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

print(f"Nombre total d'enregistrements dans Bronze: {bronze_df.count()}")
print("\nAperçu des données Bronze:")
bronze_df.show(10, truncate=False)

In [ ]:
# Statistiques par bâtiment et type de capteur
print("\nStatistiques par bâtiment:")
bronze_df.groupBy("building").agg(
    count("*").alias("total_records"),
    countDistinct("device_id").alias("unique_devices")
).show()

print("\nStatistiques par type de capteur:")
bronze_df.groupBy("type").agg(
    count("*").alias("total_records"),
    avg("value").alias("avg_value"),
    min("value").alias("min_value"),
    max("value").alias("max_value")
).show()

In [ ]:
# Détection des anomalies
print("\nEnregistrements avec anomalies détectées:")
anomalies = bronze_df.filter(col("anomaly_detected") == True)
print(f"Total anomalies: {anomalies.count()}")
anomalies.groupBy("type", "building").count().orderBy(desc("count")).show()

## 11. Historique Delta Lake

Vérification des versions et de l'historique des transactions

In [ ]:
# Historique des transactions Delta
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, BRONZE_PATH)
print("\nHistorique des transactions Delta:")
delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(10, truncate=False)

## 12. Arrêt du Streaming

Arrêt propre du query de streaming

In [ ]:
# Arrêter le streaming query
query.stop()
print("Streaming query arrêté")

# Vérifier l'état
time.sleep(2)
print(f"Is Active: {query.isActive}")

## 13. Nettoyage (Optionnel)

In [ ]:
# Décommenter pour nettoyer les données de test
# import shutil
# shutil.rmtree(BASE_PATH, ignore_errors=True)
# print(f"Répertoire {BASE_PATH} supprimé")

## Concepts Clés Illustrés

### 1. **Streaming Structuré**
- Traitement de flux comme des tables infinies
- API unifiée pour batch et streaming

### 2. **Checkpointing**
- Sauvegarde de l'état du streaming
- Reprise après échec exactement au même point
- Garantie de tolérance aux pannes

### 3. **Mode de Sortie : Append**
- Seules les nouvelles lignes sont écrites
- Idéal pour les données immuables (Bronze)

### 4. **Triggers**
- `processingTime` : traitement à intervalles réguliers (10s)
- Autres options : `once`, `continuous`

### 5. **Delta Lake (Bronze)**
- Transactions ACID
- Versioning et time travel
- Données brutes avec métadonnées d'ingestion

### 6. **Architecture Médaillon - Bronze**
- Stockage des données brutes
- Transformations minimales (validation, filtrage)
- Source de vérité pour les couches suivantes (Silver, Gold)